# DK ↔ EP Title Embeddings

Embed translated Danish Folketinget roll-call titles and European Parliament vote titles with `all-mpnet-base-v2`, save vectors, and inspect whether the two corpora sit close or far in embedding space.

Spec: `docs/superpowers/specs/2026-09-21-dk-ep-title-embeddings-design.md`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path("..")
DK_PATH = PROJECT_DIR / "data" / "temp_data" / "parliament" / "roll_calls_translated.csv"
EP_DATA_DIR = PROJECT_DIR / "EP" / "EP-data"
EP_PERIODS = ["2009-2014", "2014-2019", "2019-2024", "2024-2029"]
OUT_DIR = PROJECT_DIR / "data" / "temp_data" / "embeddings"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
RANDOM_STATE = 42

In [ ]:
df_dk = pd.read_csv(DK_PATH)
df_dk = df_dk.dropna(subset=["sag_titel_en"]).copy()
df_dk["sag_titel_en"] = df_dk["sag_titel_en"].astype(str).str.strip()
df_dk = df_dk[df_dk["sag_titel_en"].str.len() > 0].reset_index(drop=True)

print(f"DK titles: {len(df_dk):,}")
print(df_dk[["afstemningid", "sag_titel", "sag_titel_en"]].head(3))

In [ ]:
ep_frames = []
for period in EP_PERIODS:
    path = EP_DATA_DIR / period / "ep_votes.csv"
    if not path.exists():
        print(f"Missing: {path}")
        continue
    df = pd.read_csv(path)
    df["period"] = period
    if "is_main" in df.columns:
        before = len(df)
        df = df[df["is_main"] == True].copy()
        print(f"{period}: {before:,} → {len(df):,} (is_main=True)")
    else:
        print(f"{period}: {len(df):,} (no is_main column; keeping all)")
    ep_frames.append(df)

df_ep = pd.concat(ep_frames, ignore_index=True)
df_ep = df_ep.dropna(subset=["display_title"]).copy()
df_ep["display_title"] = df_ep["display_title"].astype(str).str.strip()
df_ep = df_ep[df_ep["display_title"].str.len() > 0].reset_index(drop=True)

print(f"EP titles: {len(df_ep):,}")
print(df_ep[["id", "period", "display_title"]].head(3))

## Task 2: Encode titles and save embeddings + meta

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME)

dk_texts = df_dk["sag_titel_en"].tolist()
ep_texts = df_ep["display_title"].tolist()

dk_emb = model.encode(
    dk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
ep_emb = model.encode(
    ep_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

dk_emb = np.asarray(dk_emb, dtype=np.float32)
ep_emb = np.asarray(ep_emb, dtype=np.float32)

print(MODEL_NAME)
print(f"DK embeddings: {dk_emb.shape}")
print(f"EP embeddings: {ep_emb.shape}")
assert dk_emb.shape[0] == len(df_dk)
assert ep_emb.shape[0] == len(df_ep)
assert np.allclose(np.linalg.norm(dk_emb, axis=1), 1.0, atol=1e-3)
assert np.allclose(np.linalg.norm(ep_emb, axis=1), 1.0, atol=1e-3)

In [ ]:
dk_meta_cols = [c for c in ["afstemningid", "sag_titel", "sag_titel_en", "dato"] if c in df_dk.columns]
ep_meta_cols = [c for c in ["id", "period", "display_title", "timestamp", "procedure_reference"] if c in df_ep.columns]

df_dk[dk_meta_cols].to_csv(OUT_DIR / "dk_title_meta.csv", index=False)
df_ep[ep_meta_cols].to_csv(OUT_DIR / "ep_title_meta.csv", index=False)
np.save(OUT_DIR / "dk_title_embeddings.npy", dk_emb)
np.save(OUT_DIR / "ep_title_embeddings.npy", ep_emb)

print(f"Wrote artifacts to {OUT_DIR.resolve()}")
for name in [
    "dk_title_embeddings.npy",
    "dk_title_meta.csv",
    "ep_title_embeddings.npy",
    "ep_title_meta.csv",
]:
    print(f"  {name}: {(OUT_DIR / name).stat().st_size:,} bytes")

In [ ]:
dk_emb_reload = np.load(OUT_DIR / "dk_title_embeddings.npy")
ep_emb_reload = np.load(OUT_DIR / "ep_title_embeddings.npy")
dk_meta_reload = pd.read_csv(OUT_DIR / "dk_title_meta.csv")
ep_meta_reload = pd.read_csv(OUT_DIR / "ep_title_meta.csv")

assert dk_emb_reload.shape == dk_emb.shape
assert ep_emb_reload.shape == ep_emb.shape
assert len(dk_meta_reload) == dk_emb.shape[0]
assert len(ep_meta_reload) == ep_emb.shape[0]
print("Reload OK")